# Session 3 — Grounding and RAG
### Making the model answer from your data

**AI Systems in Practice** · HEC Paris · Lab (1h45)

---

You are the AI lead at **Velora**, an e-bike retailer. Support gets about 2 000 emails a week. The most expensive ones are about the warranty.

Velora has 22 documents that contain every answer. The model has never seen them.

Today you connect the model to these documents. Then you measure where the system fails.

Steps:

1. Ask the model 20 Velora questions with no documents. **This is designed to fail.**
2. Build a retrieval pipeline: chunk, embed, search, generate.
3. Split accuracy into a retrieval rate and a generation rate.
4. Make one change. Write your prediction before you run it.
5. Test questions that have no answer in the documents.

**Key idea:**

> RAG does not make the model knowledgeable. It moves the problem from generation to retrieval. Retrieval is where it fails.

| | | |
|---|---|---|
| 0. Setup | 10 min | data, model, API key |
| A. The bare model | 12 min | accuracy and refusal rate, no documents |
| B. The naive pipeline | 25 min | TODO 1–2, your v1 accuracy |
| C. Splitting the failure | 25 min | retrieval rate and generation rate |
| D. One repair | 20 min | TODO 3–4, predict before you measure |
| E. Questions with no answer | 10 min | TODO 5, your refusal rate |
| F. Bonus | 10 min | reranking, or the whole corpus in one prompt |

**Deliverable:** this executed notebook (end of session) and a one-page report (23:59 tonight). See the last cell.


## 0. Setup

Install the packages, download the data, load the documents, and connect to the model.


In [ ]:
!pip install -q numpy pandas openai

import json, os, re, urllib.request, zipfile
from getpass import getpass

import numpy as np
import pandas as pd
from openai import OpenAI

print("ready")

In [ ]:
# download the data from GitHub
BASE = "https://raw.githubusercontent.com/vorpax/velora-labs/refs/heads/main/s3/"

FILES   = ["questions_dev.json"]
FOLDERS = ["velora_kb"]      # published as a .zip, unpacked here

for name in FILES:
    urllib.request.urlretrieve(BASE + name, name) # download the file
for name in FOLDERS:
    urllib.request.urlretrieve(BASE + name + ".zip", name + ".zip") # download the zip
    zipfile.ZipFile(name + ".zip").extractall() # unzip/unpack it

print("downloaded:", sorted(p for p in os.listdir() if not p.startswith(".")))

In [ ]:
# Load the 22 documents and the 20 questions.
# Each document file has a header (metadata) and a body, separated by "---".
QUESTIONS = json.load(open("questions_dev.json"))

DOCS = []
for name in sorted(os.listdir("velora_kb")):
    _, front, body = open(f"velora_kb/{name}", encoding="utf-8").read().split("---", 2) # split header and body
    meta = dict(line.split(":", 1) for line in front.strip().splitlines()) # header lines -> dict
    DOCS.append({k.strip(): v.strip() for k, v in meta.items()} | {"body": body.strip()}) # one dict per document

print(f"loaded {len(DOCS)} documents and {len(QUESTIONS)} questions")

In [ ]:
print(f"{len(DOCS)} documents, {sum(len(d['body']) for d in DOCS):,} characters "
      f"(~{sum(len(d['body']) for d in DOCS) // 4:,} tokens)")
print(f"{len(QUESTIONS)} questions, of which "
      f"{sum(1 for q in QUESTIONS if not q['answerable'])} have no answer in the corpus")
print()
for d in DOCS[:4]:
    print(f"  {d['id']:32s} {d['category']:10s} effective={d['effective']}")
print("  ...")

### The `gold_chunk_ids` field

Each question has a `gold_chunk_ids` field: the document sections that contain the answer.

With it, you can tell two failures apart: the search did not find the right passage, or the model did not use it. They have different fixes.

```
accuracy  ≈  retrieval_rate  ×  generation_rate
```

You will compute both in Part C.

In [ ]:
# Two models, one key.
# `model` writes the answers. `emb_model` turns text into vectors for the search (Part B).

AI_CONFIG = {
    "base_url": "https://openrouter.ai/api/v1",          # OpenRouter endpoint
    "model": "google/gemini-3.8-flash",       # generation
    "emb_model": "gemini-embedding-001",    # retrieval
    "price_in": 0.75, "price_out": 3.75,    # EUR per 1M tokens
}

print(f"generation: {AI_CONFIG['model']} at {AI_CONFIG['price_in']} EUR per 1M input tokens")
print(f"retrieval : {AI_CONFIG['emb_model']}")

In [ ]:
# OpenAI client, pointed at the OpenRouter API. Retries on network errors.
client = OpenAI(
    api_key=getpass("Generation API key: ").strip(),
    base_url=AI_CONFIG["base_url"],
    max_retries=5,
)

print("client ready")

## A. The bare model · 12 min

Ask the model the 20 Velora questions with no documents. This is your starting point.

**This part is designed to fail.** The model has never seen Velora's documents. Do not try to fix the prompt.

Two numbers come out:

- **accuracy**: expect 5 to 15 %.
- **refusal rate**: the share of answers that say "I don't know". Expect close to 0 %.

The second number matters most. The model is wrong, and it does not say so.

In [ ]:
# The prompt has no documents. Only the question.
BARE_PROMPT = """You are a support assistant for Velora, a European e-bike retailer.

Question: {question}

Answer in at most three sentences.""" # we create a prompt template with a placeholder for the question

# One call per question, one row per answer.
rows = []
for q in QUESTIONS:
    r = client.chat.completions.create( # call the model, same function as Session 1 and Session 2
        model=AI_CONFIG["model"], # We use the AI_CONFIG dictionary to get the model name
        temperature=0, # temperature=0 means somewhat deterministic answers (Session 1 for reference)
        messages=[{"role": "user",
                   "content": BARE_PROMPT.format(question=q["question"])}], # we do format the prompt
    )
    rows.append({"id": q["id"], # unique question ID
                 "question": q["question"], # the question text
                 "answerable": q["answerable"], # True if the documents contain the answer
                 "answer": (r.choices[0].message.content or "").strip(),
                 "in_tokens": r.usage.prompt_tokens}) # the number of input tokens

BARE = pd.DataFrame(rows) # Use a DataFrame to store the results, one row per question
print(f"{len(BARE)} answers, {BARE.in_tokens.sum()} input tokens")
print()
for _, row in BARE.head(3).iterrows():
    print("=" * 74)
    print(f"{row.id}  {row.question}")
    print(f"  -> {row.answer[:260]}")

In [ ]:
# We print all the answers, so you can see the model's behaviour on all questions.

for _, row in BARE.iterrows():
    print("=" * 74)
    print(f"{row.id}  {row.question}")
    print(f"  -> {row.answer[:260]}")
    print(f"  -> answerable={row.answerable}, in_tokens={row.in_tokens}")


In [ ]:
# The same call, as a function, for convenience.
def ask_model(prompt):
    r = client.chat.completions.create(
        model=AI_CONFIG["model"], temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return ((r.choices[0].message.content or "").strip(),
            r.usage.prompt_tokens, r.usage.completion_tokens)

text, tin, tout = ask_model("Reply with exactly the word: ready")
print(f"{text!r}   ({tin} input tokens, {tout} output tokens)")

In [ ]:
# ------------------------------------------------- refusal detection (pre-written)
# A refusal is detected by keywords in the answer text. 
# No model grades another model here. That method has known biases. It comes in session 5.

REFUSAL_MARKERS = (
    "i don't know", "i do not know", "not stated", "no information",
    "cannot determine", "can't determine", "unable to determine", "not specified",
    "i don't have", "i do not have", "not in the provided", "not found in",
    "the documents do not", "the context does not", "insufficient information",
    "cannot answer", "can't answer", "no answer",
)

def is_refusal(text):
    t = " ".join(str(text).lower().split())
    return any(marker in t for marker in REFUSAL_MARKERS)

# Quick test of the function.
assert is_refusal("I don't have information about that.")
assert is_refusal("The documents do not mention a parental leave policy.")
assert not is_refusal("The battery is covered for 24 months or 500 charge cycles.")
print("refusal detector: ok")

In [ ]:
# --------------------------------------------- how often is the bare model right?
# Grading by keywords, on the questions the documents can answer.
# This is a rough grade. Part A only needs an order of magnitude.

BARE_KEYS = {
    "Q01": ["24", "500"], "Q02": ["no", "500"], "Q03": ["7"], "Q04": ["50"],
    "Q05": ["no", "not covered"], "Q06": ["22"], "Q07": ["89"], "Q08": ["24"],
    "Q09": ["commercial manager"], "Q11": ["10"], "Q12": ["replace"],
    "Q13": ["740"], "Q14": ["yes", "30"], "Q15": ["96"], "Q17": ["no"],
    "Q18": ["1078"], "Q19": ["500", "14"],
} # all keywords must appear in the answer for it to count as correct

correct = []
for _, row in BARE.iterrows():
    keys = BARE_KEYS.get(row.id, [])
    text = " ".join(row.answer.lower().split())
    correct.append(bool(keys) and all(k in text for k in keys) if row.answerable else None) # None for unanswerable questions

BARE["correct"] = correct
BARE["refused"] = BARE.answer.apply(is_refusal)
ans, unans = BARE[BARE.answerable], BARE[~BARE.answerable] # answerable / unanswerable questions

bare_acc = ans.correct.mean()
bare_ref_ans = ans.refused.mean()
bare_ref_unans = unans.refused.mean()

print(f"answerable questions      : {len(ans)}")
print(f"BARE ACCURACY             : {bare_acc:.0%}    <- number 1 for your report")
print(f"refusal rate, answerable  : {bare_ref_ans:.0%}")
print(f"REFUSAL RATE, UNANSWERABLE: {bare_ref_unans:.0%}    <- number 2, the important one")
print(f"\nThe model refused {unans.refused.sum()} times on the {len(unans)} questions "
      f"that have no answer in Velora's documents.")
for _, row in unans.iterrows():
    print(f"\n  {row.id}  {row.question}\n    -> {row.answer[:220]}")

**What just happened**

Twenty fluent, professional answers. All guesses. Most wrong.

- Accuracy near zero is expected. Documents will fix it.
- A refusal rate near zero on questions with no answer is not fixed by documents.

*If your accuracy is higher than expected*: the model guessed a common market value. 24 months is a standard battery warranty.

## B. The naive pipeline · 25 min

Four stages. Each one can fail on its own:

```
   documents ──chunk──> chunks ──embed──> vectors
                                             │
   question ──embed──> vector ──cosine──> top-k ──> prompt ──> answer
```

You make two decisions:

- **TODO 1**: how to cut the documents into chunks.
- **TODO 2**: how many chunks go into the prompt (k), and the context budget.


In [ ]:
# =============================== TODO 1 ======================================
# How do you cut the 22 documents into pieces that can be searched?
#
# Two strategies are implemented below. Pick one with CHUNK_STRATEGY and set its
# parameters.
#
#   "fixed"    cut every N characters, with an overlap. Simple and predictable.
#              It can cut through a sentence or a table.
#   "section"  cut on markdown headings. Follows the document structure.
#              Chunk sizes vary a lot.
#
# Before you run the cell, write down which questions your choice will break, and why.


CHUNK_STRATEGY = "fixed"      # <- "fixed" or "section"
CHUNK_SIZE     = 900      # <- characters, "fixed" only
CHUNK_OVERLAP  =  20      # <- characters, "fixed" only

WHAT_THIS_BREAKS = "iana - kamura aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa"    # <- a string: which questions will fail, and why

assert CHUNK_STRATEGY in ("fixed", "section")
assert isinstance(WHAT_THIS_BREAKS, str) and len(WHAT_THIS_BREAKS) > 80, \
    "Name the questions your choice will break, and why."
print(f"strategy={CHUNK_STRATEGY} size={CHUNK_SIZE} overlap={CHUNK_OVERLAP}")
print(WHAT_THIS_BREAKS)

### Section bookkeeping

Each chunk is tagged with the `section_ids` it covers. This lets you compare retrieved chunks to `gold_chunk_ids` in Part C.

A fixed-size chunk covers a section only if it contains at least half of it. Otherwise recall would be too high.


In [ ]:
# find the sections of a document, one per markdown heading
def sections(body):
    """[(slug, heading, text, start, end)] over a document body, in order."""
    out, slug, head, buf, start, pos = [], None, None, [], 0, 0
    for line in body.splitlines(keepends=True):
        if line.startswith("#"): # a markdown heading
            if slug is not None: # if we have a previous section, save it
                out.append((slug, head, "".join(buf).strip(), start, pos))
            head = line.lstrip("#").strip() # the heading text, without the leading #s
            slug = re.sub(r"[^a-z0-9]+", "-", head.lower()).strip("-") # slug is a URL-friendly version of the heading
            buf, start = [], pos # reset the buffer and start position for the new section
        else:
            buf.append(line) # add the line to the current section's buffer
        pos += len(line) # increment the position counter by the length of the line
    if slug is not None: # if we have a previous section, save it
        out.append((slug, head, "".join(buf).strip(), start, pos)) # save the last section
    return out

print(DOCS[0]["id"], "->", [s[0] for s in sections(DOCS[0]["body"])])

In [ ]:
# the chunker
SECTION_PACK = 900        # "section": pack sections until a chunk reaches about this size (characters)
OVERLAP_FRACTION = 0.5    # "fixed": a chunk covers a section only if it contains this fraction of it

# "section": pack whole sections into chunks of about SECTION_PACK characters
def chunk_section(doc, secs, common):
    out, ids, txt = [], [], []
    for slug, head, text, _s, _e in secs: # iterate over the sections of the document
        if not text: # skip empty sections
            continue
        ids.append(f"{doc['id']}#{slug}") # add the section ID to the list of IDs for this chunk
        txt.append(f"{head}\n\n{text}") # add the section text to the current chunk
        if sum(len(x) for x in txt) >= SECTION_PACK: # chunk is full: save it, start a new one
            out.append({**common, "section_ids": ids, "text": "\n\n".join(txt)}) # save the chunk with metadata and section IDs
            ids, txt = [], [] 
    if txt: # save the last chunk
        out.append({**common, "section_ids": ids, "text": "\n\n".join(txt)})
    return out

# "fixed": cut the body into windows of `size` characters, moving forward by (size - overlap)
def chunk_fixed(doc, secs, common, size, overlap):
    out = [] # a list to hold the chunks
    for start in range(0, max(1, len(doc["body"])), max(1, size - overlap)): # iterate over the document body in steps of (size - overlap)
        piece = doc["body"][start:start + size] # extract a piece of the document body of length `size`
        if not piece.strip(): # skip empty pieces
            continue
        end = start + len(piece) # calculate the end position of the current piece
        ids = [f"{doc['id']}#{slug}" for slug, _h, _t, s0, s1 in secs # the window covers a section
               if min(end, s1) - max(start, s0) >= OVERLAP_FRACTION * (s1 - s0 or 1)] # only if it holds enough of it
        out.append({**common, "section_ids": ids, "text": piece.strip()}) # save the chunk with metadata and section IDs
    return out

# chunk every document with the chosen strategy, then number the chunks
def chunk_corpus(docs, strategy, size, overlap):
    chunks = [] # a list to hold all the chunks from all documents
    for d in docs: # iterate over each document
        secs = sections(d["body"])
        common = {"doc_id": d["id"], "doc_title": d["title"], "effective": d["effective"]} # metadata copied into each chunk
        if strategy == "section":
            chunks += chunk_section(d, secs, common)
        elif strategy == "fixed":
            chunks += chunk_fixed(d, secs, common, size, overlap)
    for i, c in enumerate(chunks): # number the chunks sequentially
        c["chunk_id"] = i
    return chunks


print("chunking functions defined")


In [ ]:
CHUNKS = chunk_corpus(DOCS, CHUNK_STRATEGY, CHUNK_SIZE or 0, CHUNK_OVERLAP or 0) # chunk the documents using the chosen strategy and parameters
sizes = [len(c["text"]) for c in CHUNKS] # get the number of characters inside of all chunks
print(f"{len(CHUNKS)} chunks: min {min(sizes)}, median {int(np.median(sizes))}, "
      f"max {max(sizes)} chars — ~{sum(sizes) // 4:,} tokens indexed")

### Embedding: turning text into vectors

An embedding turns text into a vector. Close meanings give close vectors. Search becomes a dot product.

You do not build the model. You call an API. The cell below caches the results, so no text is paid for twice.

Read the printed cosine. It shows why this architecture works.

In [ ]:
# embedding, with a cache: each text is sent to the API only once

_VEC = {} # a dictionary to cache the embeddings of texts


def embed(texts, batch=100):
    """A list of strings -> an (n, dim) matrix, one L2-normalised row each."""
    new = [t for t in dict.fromkeys(texts) if t not in _VEC] # remove duplicates and already cached texts
    for i in range(0, len(new), batch):
        r = client.embeddings.create(model=AI_CONFIG["emb_model"], input=new[i:i + batch]) # request embeddings for a batch of texts
        for t, d in zip(new[i:i + batch], r.data): # cache the embeddings in the _VEC dictionary
            _VEC[t] = d.embedding # store the embedding vector for the text t in the _VEC dictionary
    M = np.array([_VEC[t] for t in texts], dtype=np.float32) # create a matrix of embeddings for the input texts
    # Rows are normalised once, so a dot product below is a cosine similarity.
    return M / np.linalg.norm(M, axis=1, keepdims=True) # L2-normalise each row of the matrix so that the dot product between two rows is the cosine similarity between the corresponding texts.


v = embed(["How long is the battery guaranteed?",
           "Cell coverage period: 24 months or 500 complete charge cycles."])
print(f"{v.shape[1]} dimensions, {len(_VEC)} vectors cached")
print(f"cosine between those two sentences = {v[0] @ v[1]:.2f}")


### Why the index is a numpy array

A few hundred chunks. One matrix multiply. Under a millisecond.

A vector database gives approximate search over millions of vectors. It costs a dependency, a service, and an index to keep in sync. When that trade flips is a question for session 6. Measure first.

In [ ]:
# ------------------------------------------------------ the index (pre-written)
MATRIX = embed([c["text"] for c in CHUNKS]) # embed all the chunk texts into a matrix, one row per chunk
print(f"indexed {MATRIX.shape[0]} chunks x {MATRIX.shape[1]} dims") # print the shape of the embedding matrix

# Embed all twenty questions now, in ONE batched call.
# search() embeds its query each time it runs, and it runs about 200 times today.
# With the cache, each question is embedded once.

embed([q["question"] for q in QUESTIONS]) # embed all the questions into the cache
print(f"{len(_VEC)} vectors cached, {len(QUESTIONS)} of them queries")

def search(question, k=5):
    """Exact cosine search. Vectors are L2-normalised, so a dot product IS the cosine."""
    scores = MATRIX @ embed([question])[0] # compute the cosine similarity scores between the question embedding and all chunk embeddings
    top = np.argsort(-scores)[:k] # get the indices of the top k chunks with the highest scores
    return [{**CHUNKS[i], "score": float(scores[i])} for i in top] # return the top k chunks with their metadata and scores

for hit in search("How long is the battery guaranteed?", k=3): # search for the top 3 chunks relevant to the question
    print(f"\n  {hit['score']:.3f}  {hit['doc_id']}  {hit['section_ids']}")
    print(f"         {hit['text'][:110]!r}")

### Look at the result before you go on

The query was "How long is the battery guaranteed?". Three things to notice:

1. **"Guaranteed" appears in none of the documents.** They say *cell coverage period*. The search found them anyway, with no synonym list. This is what embeddings give you over keyword search.

2. **A helmet chunk is probably near the top.** Its warranty document has the same structure as the battery one. Similar is not relevant.

3. **Two documents give different answers to this question.** The ranking does not tell you. One is outdated.

Similar is not relevant. Relevant is not always enough.

In [ ]:
# =============================== TODO 2 ======================================
# How many chunks go into the prompt, in what order, and what do you drop?
#
# Each extra chunk raises the chance that the answer is in the context. It also raises
# the cost, and adds one more passage the model has to ignore.
#
# Three settings:
#
#   K              how many chunks you retrieve
#   ORDER          "score" (most similar first) or "reverse" (most similar last).
#                  Models do not read a long context evenly. Position matters, and
#                  it is free to change.
#   MAX_CHARS      the context budget. If the K chunks are larger, some are dropped.
#
# Then write one sentence that justifies your K. "More is better" is not enough.
# Say what you expect to gain with K+3, and what it costs at 400 requests a day.

K          = ...     # <- an integer between 1 and 20, how many chunks to retrieve
ORDER      = ...      # <- "score" or "reverse"
MAX_CHARS  = ...   # <- the context budget, in characters
WHY_K      = ...      # <- one sentence: what K+3 would gain, and what it would cost

assert isinstance(K, int) and 1 <= K <= 20
assert ORDER in ("score", "reverse")
assert isinstance(MAX_CHARS, int) and MAX_CHARS >= 500
assert isinstance(WHY_K, str) and len(WHY_K) > 60
print(f"K={K} order={ORDER} budget={MAX_CHARS} chars")
print(WHY_K)

In [ ]:
# ------------------------------------------------- the RAG prompt (pre-written)
# A system prompt with a slot for the retrieved passages. Same idea as in session 2,
# with the context in place of the email body. The citation instruction matters:
# a citation is what lets the reader check the answer.
RAG_PROMPT = """You are a support assistant for Velora, a European e-bike retailer.

Answer the question using ONLY the numbered context passages below. If the passages do
not contain the answer, say exactly: "I don't know — this is not in the Velora
documents."

Cite the passage numbers you used, in square brackets, at the end of your answer.

CONTEXT
{context}

QUESTION
{question}

ANSWER (at most three sentences, with citations):"""

def build_context(hits, order, max_chars):
    """Assemble the retrieved chunks into the prompt slot, within the budget."""
    kept, used = [], 0
    for h in hits:                       # drop the least similar chunks first
        if used + len(h["text"]) > max_chars and kept:
            break
        kept.append(h)
        used += len(h["text"])
    ordered = kept if order == "score" else list(reversed(kept))
    return "\n\n".join(
        f"[{i}] {h['doc_title']} (effective {h['effective']})\n{h['text']}"
        for i, h in enumerate(ordered, 1)), kept

hits = search(QUESTIONS[0]["question"], k=K)
context, kept = build_context(hits, ORDER, MAX_CHARS)
print(f"{len(context)} chars of context from {len(kept)} chunks. What gets sent:\n")
print(RAG_PROMPT.format(context=context, question=QUESTIONS[0]["question"])[:700])

In [ ]:
# ------------------------------------------------------- the scorer (pre-written)
# Grading by keywords against the known answer. Rough on purpose. The alternative is a
# model grading a model. That method has known biases and comes in session 5.

ANSWER_KEYS = {
    "Q01": ["24", "500"], "Q02": ["500"], "Q03": ["7"], "Q04": ["50"],
    "Q05": ["not cover"], "Q06": ["22"], "Q07": ["89"], "Q08": ["24"],
    "Q09": ["commercial manager"], "Q11": ["10"], "Q12": ["waive", "replace"],
    "Q13": ["740"], "Q14": ["resaleable"], "Q15": ["96"], "Q17": ["no"],
    "Q18": ["1078"], "Q19": ["14"],
} # all keywords must appear in the answer for it to count as correct

def graded(qid, text):
    """None for unanswerable questions — they are scored in Part E instead."""
    keys = ANSWER_KEYS.get(qid) # get the keywords for the question ID
    if keys is None: # if the question ID is not in the ANSWER_KEYS dictionary, it means the question is unanswerable
        return None
    t = " ".join(str(text).lower().split()) # normalize the text to lowercase and remove extra whitespace
    return any(k.lower() in t for k in keys)

assert graded("Q07", "Delivery to Switzerland is EUR 89. [2]")
assert not graded("Q07", "Delivery to Switzerland is EUR 49. [2]")
assert graded("Q10", "Sixteen weeks of parental leave.") is None
print("scorer: ok")

In [ ]:
# ---------------------------------------------------------------- run v1
# Retrieve, assemble, generate, score. Twenty questions, one row each.
rows = []
for q in QUESTIONS:
    hits = search(q["question"], k=K) # search for the top K chunks relevant to the question
    context, kept = build_context(hits, ORDER, MAX_CHARS) # assemble the context from the retrieved chunks, within the budget
    text, tin, tout = ask_model(RAG_PROMPT.format(context=context, question=q["question"])) # ask the model to generate an answer based on the context and question
    gold = set(q["gold_chunk_ids"]) # the set of gold chunk IDs for the question
    retrieved = {sid for h in kept for sid in h["section_ids"]} # the set of section IDs retrieved in the context
    rows.append({
        "id": q["id"], # unique question ID
        "question": q["question"],  # the question text
        "traps": ",".join(q["traps"]), # the traps associated with the question
        "answerable": q["answerable"], "answer": text, # the model's answer
        "correct": graded(q["id"], text), "refused": is_refusal(text), # whether the model refused to answer
        
        
        "recall_any": bool(gold & retrieved) if gold else None, # recall_any: at least one gold section was retrieved.
        "recall_all": gold <= retrieved if gold else None, # recall_all: every gold section was retrieved (some questions need two). 
        "n_gold": len(gold), # the number of gold sections for the question
        "n_kept": len(kept),  # the number of chunks kept in the context
        "context_chars": len(context), # the number of characters in the context
        "in_tokens": tin, # the number of input tokens sent to the model
        "cost_eur": tin * AI_CONFIG["price_in"] / 1e6 + tout * AI_CONFIG["price_out"] / 1e6, # the cost of the request in EUR
    })

V1 = pd.DataFrame(rows)
ansv1 = V1[V1.answerable]
v1_acc = ansv1.correct.mean()

print(f"questions    : {len(V1)} ({len(ansv1)} answerable)")
print(f"V1 ACCURACY  : {v1_acc:.0%}    <- your number for the class discussion")
print(f"mean context : {V1.context_chars.mean():.0f} chars, "
      f"{V1.in_tokens.mean():.0f} input tokens, {V1.cost_eur.mean() * 1000:.2f} EUR/1k requests")
print()
print(V1[["id", "traps", "correct", "recall_any", "recall_all"]].to_string(index=False))

### Class discussion — 5 minutes (optional)

Three pairs read out three numbers: their **chunk size**, their **K**, and their **v1 accuracy**.

The accuracies will differ by 15 to 25 points. Same corpus, same questions, same model. The only difference is a chunking decision.

Write the spread on the board.

## C. Splitting the failure · 25 min

One accuracy number does not tell you what to fix. Split it:

- **retrieval rate `r`**: how often the right passage was in the context.
- **generation rate `g`**: how often the model answered correctly, **when** the passage was there.

```
accuracy  ≈  r × g
```

> **`r` is a ceiling. No prompt work can go above it.**

If the passage is not in the context, no instruction, no phrasing and no bigger model can recover it.

In [ ]:
# ------------------------------------------------- the split
ans_only = V1[V1.answerable].copy()

r_any = ans_only.recall_any.mean()
r_all = ans_only.recall_all.mean()
found = ans_only[ans_only.recall_any]
g = found.correct.mean()

print(f"RETRIEVAL RATE  r  (any gold chunk retrieved) : {r_any:.0%}")
print(f"                   (ALL gold chunks retrieved): {r_all:.0%}")
print(f"GENERATION RATE g  (correct GIVEN retrieved)  : {g:.0%}   on {len(found)} questions")
print(f"                                   r x g      = {r_any*g:.0%}")
print(f"observed accuracy                             : {ans_only.correct.mean():.0%}")
print()
print("The gap between r x g and observed accuracy comes from questions answered correctly")
print("WITHOUT the gold chunk: the model guessed, or knew a common market value.")
print("You cannot rely on those.")

In [ ]:
# ------------------------------------------------- recall@k, as a curve
# Recall depends on k. The shape of the curve tells you if a bigger k helps, or if you
# need a different chunker.
curve = []
for kk in (1, 2, 3, 5, 8, 12, 20):
    hit_any, hit_all = 0, 0
    n = 0
    for q in QUESTIONS:
        gold = set(q["gold_chunk_ids"])
        if not gold:
            continue
        n += 1
        got = {sid for h in search(q["question"], k=kk) for sid in h["section_ids"]}
        hit_any += bool(gold & got)
        hit_all += gold <= got
    curve.append({"k": kk, "recall_any": hit_any / n, "recall_all": hit_all / n,
                  "added_tokens": int(kk * np.median([len(c["text"]) for c in CHUNKS]) / 4)})

CURVE = pd.DataFrame(curve).set_index("k")
display(CURVE.style.format({"recall_any": "{:.0%}", "recall_all": "{:.0%}",
                            "added_tokens": "{:,.0f}"}))

r3 = CURVE.loc[3, "recall_any"]
r8 = CURVE.loc[8, "recall_any"]
print(f"\nrecall@3 = {r3:.0%}   recall@8 = {r8:.0%}   "
      f"difference = {(r8-r3)*100:+.0f} points")
print(f"At g = {g:.0%}, that difference is worth {(r8-r3)*g*100:+.0f} points of accuracy.")

# What those points cost at Velora's volume.
extra_tokens = CURVE.loc[8, "added_tokens"] - CURVE.loc[3, "added_tokens"]
per_year = 400 * 365 * extra_tokens * AI_CONFIG["price_in"] / 1e6
print(f"Cost of k=3 -> k=8: ~{extra_tokens:,} extra input tokens per request, "
      f"~{per_year:.0f} EUR/year at 400 requests/day.")
print("Keep these two numbers together: the accuracy points and the euros.")

### The five failure modes

Every failure in your table is one of these. Each has a different fix. Telling them apart is the skill.

| | what happened | how to recognise it |
|---|---|---|
| **F1** | The answer is not in the corpus, but the system answered anyway | `answerable` is false and `refused` is false |
| **F2** | The answer is in the corpus, but retrieval missed it | `recall_any` is false |
| **F3** | The answer was cut by a chunk boundary | the chunk is retrieved, but its header or its second half is missing |
| **F4** | The passage was retrieved, but the model contradicted it | `recall_any` is true, `correct` is false |
| **F5** | Two documents disagree, and the system silently picked one | both documents retrieved, one answer, no mention of the conflict |

**F5 is the one nobody expects, and the one companies actually have.** It does not look like a failure. It looks like a confident answer.

In [ ]:
# ----------------------------------------------- classify each failure
def classify(row):
    if not row.answerable:
        return "OK-refused" if row.refused else "F1"
    if not row.recall_any:
        return "F2"
    if row.correct:
        return "OK"
    if row.n_gold > 1 and not row.recall_all:
        return "F3"          # part of the answer was cut away
    return "F4"

V1["failure"] = V1.apply(classify, axis=1)
print(V1.failure.value_counts().to_string())

In [ ]:
# ------------------------------------- F5 must be searched for
# The other four classes come from the table. F5 does not. It is a failure of silence:
# two documents that disagree are both retrieved, one answer comes back, and nothing
# says that the documents contradict each other.
CONTRADICTING = {"warranty-battery-2018", "warranty-battery-2025"}
MENTIONS = ("2018", "superseded", "previous terms", "two ", "conflict", "differ")

rows = []
for _, row in V1.iterrows():
    q = next(x for x in QUESTIONS if x["id"] == row.id)
    if CONTRADICTING <= {h["doc_id"] for h in search(q["question"], k=K)}:
        rows.append({"id": row.id, "answer": row.answer[:150],
                     "mentioned_the_conflict":
                         any(w in row.answer.lower() for w in MENTIONS)})

F5 = pd.DataFrame(rows)
if len(F5):
    print("F5 candidates — BOTH warranty documents were retrieved:")
    print(F5[["id", "mentioned_the_conflict"]].to_string(index=False))
    silent = F5[~F5.mentioned_the_conflict]
    print(f"\n{len(silent)}/{len(F5)} answered from two contradicting documents "
          f"WITHOUT saying so.")
    for _, r in silent.head(2).iterrows():
        print(f"\n  {r.id}: {r.answer}")
else:
    print("No question retrieved both warranty documents at this K. Raise K and run "
          "this cell again. F5 exists in this corpus and it is worth seeing.")

In [ ]:
# ---------------------------------------- read the failures, one at a time
# The rate is not the finding. The cause is. The cause is visible in the text.
for _, row in V1[V1.failure.isin(["F2", "F3", "F4", "F1"])].head(4).iterrows():
    q = next(x for x in QUESTIONS if x["id"] == row.id)
    print("=" * 74)
    print(f"{row.id}  [{row.failure}]  traps: {row.traps}")
    print(f"  Q: {row.question}")
    print(f"  gold: {q['gold_chunk_ids']}")
    print(f"  got : {[h['section_ids'] for h in search(q['question'], k=K)][:3]}")
    print(f"  A: {row.answer[:260]}")

## D. One repair · 20 min

Change one thing. Not three. With three changes, you cannot tell which one moved the number.

Before you run the change, **write down what you expect**.

A wrong prediction gets full marks. No prediction gets zero.

In [ ]:
# ------------------------------ what do your similarity scores look like?
# Read this BEFORE you set a threshold below. A cosine score from a hosted embedding
# model does not go from 0 to 1. All text in the same language and style already scores
# high. The useful range is narrow. A round number picked by intuition is a blind guess.
_top = {q["id"]: (search(q["question"], k=1)[0]["score"], q["answerable"])
        for q in QUESTIONS}
_ans = sorted(s for s, a in _top.values() if a)
_una = sorted(s for s, a in _top.values() if not a)

print(f"answerable   (n={len(_ans)}): min {_ans[0]:.3f}  "
      f"median {_ans[len(_ans) // 2]:.3f}  max {_ans[-1]:.3f}")
print(f"unanswerable (n={len(_una)}): min {_una[0]:.3f}  "
      f"median {_una[len(_una) // 2]:.3f}  max {_una[-1]:.3f}")
print()
_overlap = sum(1 for s in _ans if s < _una[-1])
print(f"A gate only works if these two ranges do not overlap. {_overlap} answerable "
      f"question(s) score below the best unanswerable one.")
print("A threshold that refuses all three unanswerable questions also refuses those.")
print("That is your false-refusal cost. Choose SIM_THRESHOLD from these numbers.")

In [ ]:
# =============================== TODO 3 ======================================
# The refusal policy: WHERE does it live?
#
# Two options. They are not the same:
#
#   in the PROMPT  — you ask the model to say "I don't know". Cheap and flexible.
#                    The model may ignore it. A prompt is a request, not a rule.
#   in the CODE    — you refuse before generating, when the best similarity score is
#                    below a threshold. Always enforced. But cosine similarity measures
#                    how close the topics are, not whether the passage answers the
#                    question.
#
# Set the threshold. SIM_THRESHOLD = 0.0 turns off the code-side gate. The policy then
# lives only in the prompt.
#
# Then state the cost. A threshold high enough to refuse the unanswerable questions will
# also refuse some answerable ones. Q17's answer is "no, Velora does not offer a student
# discount". A refusal there is WRONG. Say which side you prefer to err on, and for which
# Velora use case that is the right choice.

SIM_THRESHOLD   = ...     # <- float, 0.0 to disable the code-side gate
REFUSAL_POLICY  = ...      # <- a string: where the policy lives, and what it costs

assert isinstance(SIM_THRESHOLD, float) and 0.0 <= SIM_THRESHOLD < 1.0
assert isinstance(REFUSAL_POLICY, str) and len(REFUSAL_POLICY) > 80
print(f"SIM_THRESHOLD = {SIM_THRESHOLD}")
print(REFUSAL_POLICY)

In [ ]:
# =============================== TODO 4 ======================================
# ONE change. Predict the result BEFORE you run it.
#
# Pick one, or choose your own:
#
#   (a) change the chunking strategy or size
#   (b) change K
#   (c) change the context order
#   (d) add the document title and effective date at the top of each chunk, so the
#       model can see where a passage comes from
#
# Then write the prediction. It must have a NUMBER and a DIRECTION: which of accuracy,
# retrieval rate or generation rate will move, by about how much, and why.
# "It should improve things" is not a prediction. It scores zero.
#
# A wrong prediction gets full marks. No prediction gets zero.

INTERVENTION   = ""
PREDICTION     = ""

# v2 starts as a copy of v1. Change ONE line.
K2         = K
ORDER2     = ORDER
MAX_CHARS2 = MAX_CHARS
STRATEGY2  = CHUNK_STRATEGY
SIZE2      = CHUNK_SIZE
OVERLAP2   = CHUNK_OVERLAP

assert isinstance(PREDICTION, str) and any(ch.isdigit() for ch in PREDICTION), \
    "A prediction without a number is an opinion."
n_changed = sum([K2 != K, ORDER2 != ORDER, MAX_CHARS2 != MAX_CHARS,
                 STRATEGY2 != CHUNK_STRATEGY, SIZE2 != CHUNK_SIZE,
                 OVERLAP2 != CHUNK_OVERLAP])
assert n_changed == 1, f"{n_changed} things changed. Change exactly one, or you cannot " \
                       f"tell what caused the result."
print(f"intervention: {INTERVENTION}")
print(f"prediction  : {PREDICTION}")

In [ ]:
# ---------------------------------------------------------------- run v2
# Same twenty questions, same loop as `run v1`, with ONE thing changed.
# Compare the two cells side by side. The difference is your change.
CHUNKS = chunk_corpus(DOCS, STRATEGY2, SIZE2 or 0, OVERLAP2 or 0)
MATRIX = embed([c["text"] for c in CHUNKS])
print(f"re-indexed: {len(CHUNKS)} chunks")

rows = []
for q in QUESTIONS:
    hits = search(q["question"], k=K2)
    context, kept = build_context(hits, ORDER2, MAX_CHARS2)
    text, tin, tout = ask_model(RAG_PROMPT.format(context=context, question=q["question"]))
    gold = set(q["gold_chunk_ids"])
    retrieved = {sid for h in kept for sid in h["section_ids"]}
    rows.append({
        "id": q["id"], "question": q["question"], "traps": ",".join(q["traps"]),
        "answerable": q["answerable"], "answer": text,
        "correct": graded(q["id"], text), "refused": is_refusal(text),
        "recall_any": bool(gold & retrieved) if gold else None,
        "recall_all": gold <= retrieved if gold else None,
        "n_gold": len(gold), "n_kept": len(kept), "context_chars": len(context),
        "in_tokens": tin,
        "cost_eur": tin * AI_CONFIG["price_in"] / 1e6 + tout * AI_CONFIG["price_out"] / 1e6,
    })

V2 = pd.DataFrame(rows)
V2["failure"] = V2.apply(classify, axis=1)
print(f"v2 accuracy: {V2[V2.answerable].correct.mean():.0%}")

In [ ]:
# what moved, and was it what you predicted?
a1, a2 = V1[V1.answerable], V2[V2.answerable]
comparison = pd.DataFrame({
    "v1": [a1.correct.mean(), a1.recall_any.mean(), a1.recall_all.mean(),
           a1[a1.recall_any].correct.mean(), V1.in_tokens.mean(), V1.cost_eur.mean()],
    "v2": [a2.correct.mean(), a2.recall_any.mean(), a2.recall_all.mean(),
           a2[a2.recall_any].correct.mean(), V2.in_tokens.mean(), V2.cost_eur.mean()],
}, index=["accuracy", "recall_any (r)", "recall_all", "generation (g)",
          "mean input tokens", "mean EUR/request"])
comparison["change"] = comparison.v2 - comparison.v1
display(comparison.style.format("{:.4f}"))

print(f"\nYour prediction was: {PREDICTION}")
print("\nFOR YOUR REPORT: the prediction, the result, and if they differ, what that "
      "tells you about the system. A wrong prediction, reported honestly, is worth "
      "more than a right one.")

# Which questions changed. This is where you can see the cause.
moved = V1[["id", "correct", "recall_any"]].merge(
    V2[["id", "correct", "recall_any"]], on="id", suffixes=("_v1", "_v2"))
moved = moved[(moved.correct_v1 != moved.correct_v2) |
              (moved.recall_any_v1 != moved.recall_any_v2)]
print(f"\n{len(moved)} questions changed state:")
print(moved.to_string(index=False))

## E. Questions with no answer · 10 min

Three of your twenty questions have no answer in Velora's documents.

So far you measured recall on questions written by someone who knew the corpus. Real users ask about parental leave or carbon footprint. For those questions, recall is **undefined**.

> **The most important number in a RAG system is its refusal rate on questions outside the corpus.**

Measure the other side too. **Q17's correct answer is "no, Velora does not offer a student discount."** A refusal there is a wrong answer. Refusal has a cost.

In [ ]:
# =============================== TODO 5 (optional) ============================
# When two documents disagree, which one wins, and who decides?
#
# This corpus has a battery warranty from 2025 (24 months + 500 cycles) and an old one
# from 2018 (36 months). Only one has an effective date.
#
# There are three places to put the rule. They are different:
#
#   at INDEX time   drop the old document. Simple. But you can no longer answer about
#                   a bike sold before the new terms.
#   at SEARCH time  rank documents with no effective date lower, or prefer the newest.
#   at PROMPT time  give the model both, with their dates. Ask it to say which one it
#                   used, and that a conflict exists.
#
# Pick one. Write it in CONFLICT_RULE as a sentence: the rule, where it lives, and what
# it costs. One question in the held-out set needs the OLD document. "Delete the old
# one" has a price, and you will pay it after you submit.

CONFLICT_RULE = ...       # <- a string: the rule, where it lives, and what it costs

assert isinstance(CONFLICT_RULE, str) and len(CONFLICT_RULE) > 80
print(CONFLICT_RULE)

In [ ]:
# ------------------------------------------- refusal, with the code-side gate
# The gate is three lines, and it runs BEFORE the model. Below the threshold, nothing
# is generated. This is the difference between a policy in code and a policy in a prompt.
rows = []
for q in QUESTIONS:
    hits = search(q["question"], k=K2)
    if SIM_THRESHOLD > 0 and hits[0]["score"] < SIM_THRESHOLD:
        text, gated = "I don't know — this is not in the Velora documents.", True
    else:
        context, kept = build_context(hits, ORDER2, MAX_CHARS2)
        text = ask_model(RAG_PROMPT.format(context=context, question=q["question"]))[0]
        gated = False
    rows.append({"id": q["id"], "answerable": q["answerable"],
                 "top_score": hits[0]["score"], "gated": gated,
                 "refused": is_refusal(text) or gated,
                 "correct": graded(q["id"], text), "answer": text})

E = pd.DataFrame(rows)
unans, ansE = E[~E.answerable], E[E.answerable]

print(f"REFUSAL RATE on the {len(unans)} unanswerable questions : "
      f"{unans.refused.mean():.0%}   <- the number for your report")
print(f"false refusals on answerable questions           : "
      f"{ansE.refused.sum()}/{len(ansE)} = {ansE.refused.mean():.0%}")
print(f"accuracy after gating                            : {ansE.correct.mean():.0%}")
print(f"bare model, part A, on the same questions        : {bare_ref_unans:.0%} refusal")
print()
print(E[["id", "answerable", "top_score", "gated", "refused", "correct"]]
      .to_string(index=False))
print()
print("Look at Q17. Its correct answer is 'no'. If it was refused, your threshold trades "
      "wrong answers for safety. That trade is a business decision, not a technical one.")

### The limit that no threshold fixes

**Cosine similarity measures how close two topics are. It does not measure whether a passage answers a question.**

```
"The warranty does not cover water damage."
"The warranty covers water damage."
```

Same subject, same words, same style. Opposite meanings. Their vectors are nearly identical. **An embedding does not see negation.** The retriever finds the right passage for Q05. It would find it just as well if the document said the opposite.

This is why you need a measured recall on real questions. "The search seems to work" is not evidence. The search works just as well when the corpus is wrong.

In [ ]:
# Negation test. Two sentences, opposite meanings, same vector space.
pair = ["The warranty does not cover water damage.",
        "The warranty covers water damage.",
        "The return window for accessories is fourteen days."]
V = embed(pair)
print(f"cosine('does not cover', 'covers')        = {float(V[0] @ V[1]):.3f}")
print(f"cosine('does not cover', unrelated topic) = {float(V[0] @ V[2]):.3f}")
print(f"  -> the contradicting pair is {float(V[0] @ V[1]) - float(V[0] @ V[2]):.3f} "
      f"closer than the unrelated one")
print()
print("Note the floor. Two sentences about different topics do not score zero.")
print("A hosted embedding model puts all text in the same language and style well above zero.")
print("This is why a threshold has so little room to work.")
print("Inside that narrow room, the two sentences that CONTRADICT each other are nearly identical.")
print("No threshold separates them. The distance a threshold measures is not the distance that matters.")

## F. Bonus (optional)

Pick one. A measured negative result beats an unmeasured feature.

- **The whole corpus in one prompt.** About 35 000 tokens. Put all 22 documents in the prompt and skip retrieval. It will probably be **more accurate**. Count the tokens. At 400 requests a day, compare the yearly cost to your RAG pipeline. This opens Session 6.

- **Query rewriting.** Ask the model to rewrite the question with the corpus vocabulary ("battery guarantee" → "cell coverage period") before searching. Measure recall before and after. Count the extra call.

- **Your own questions.** Write five questions a real Velora customer would ask. Run them. Count how many your system answers. This is a preview of Session 5.

In [ ]:
# Scratch space for Part F. Record the number for whatever you run. You might create new cells, the point is to experiment.

## Deliverable

One notebook and one page of text. In pairs.

1. **This notebook, executed**, with TODOs filled and outputs visible, **on Wednesday 23:59**.
2. **A one-page report, on Wednesday 23:59**, built from the numbers in this notebook :

| | |
|---|---|
| The system | v1 and v2 accuracy, with your chunking and K |
| The split | your `r` and your `g`, and which one is your ceiling |
| Failure classes | F1–F5, one real example each from your run |
| Refusal | your refusal rate on the 3 unanswerable questions, and your false refusals |
| The prediction | what you predicted in Part D, what happened, what the gap means |
| Cost | the context cost of your K at 400 requests/day |

**Grading** (one of six deliverables; your best four count for 30 % of the course):

| | |
|---|---|
| A working system, with v1/v2 accuracy measured | 5 |
| **Retrieval/generation split, interpreted** | 5 |
| Failures classified F1–F5, one example each | 4 |
| Refusal rate on the unanswerable questions | 3 |
| Part D change, predicted before it was measured | 2 |
| Context cost at Velora's volume | 1 |

Accuracy with no split scores 1 out of 5 on the second line. A wrong written prediction scores full marks on the fifth.

Every number in your report must come from a cell you ran.

---

**Before you leave: choose your capstone case today.** One paragraph: the use case, where the data comes from, and the twenty reference questions you will write.